불균형 데이터 처리 - 데이터 부족 문제 해결
- 다수 클래스의 데이터가 소수의 클래스보다 훨씬 많을 때 소수 클래스를 학습 못하는 문제
- 소수 클래스의 데이터를 추가로 만들면서 최적의 학습에 필요한 데이터 갯수를 찾는다.


- 입계값 분류로 넣고볼때 기준이되는 값
값이 높으면 양성 낮으면 음성 임계값 설정을
데이터의 성질에 따라 임계값을 어떻게 설정해야되나가 문제 ex) 암환자 설정 (임계값을 낮추는경우) 의심이들면 바로 하는 // 놓쳐도 크게 영향이 없는 것들이 이제 임계값을 높이는경우 (ex스팸메일) 중요한 메일들이 스팸메일로 가지않도록 임계값을 높여서 걸러내는

따라서
현재는 잘사용하는 방법
데이터를 분류하는방법 이 중요하다.

정확도 : 전체중 맞춘 비율 (중요도가 낮다 필요도가 낮다. )  
정밀도 : AI가 예측한 것중에 진짜로 맞는 비율  (높아야되는 것 : 스팸메일)
재현율 : 실제로 맞는데 AI가 예측한 비율 (정밀도랑 재현율이랑 반대) (높아야되는것 예시 : 암환자)  
F1 : 정밀도와 재현율의 비율

In [ ]:
# SMOTE 오버 샘플링
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

#타이타닉 로드
titanic = sns.load_dataset('titanic')
df = titanic.loc[:, ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch','fare']].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df = pd.get_dummies(df, columns=['sex'], drop_first=True , dtype = int)
X = df.drop(columns = 'survived')
y = df['survived']

# 데이터분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)
#SMOTE 객체 생성(오직 훈련데이터에만 사용)
smote = SMOTE(random_state = 42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 0이사망 1이 생존 비율 2:1
print("=============SMOTE 적용 전 클래스 분포=============")
print(f"X_train(문제개수):{X_train_resampled.shape[0]}개")
print(f"y_train(정답분포):{y_train_resampled.value_counts()}개")


rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_resampled, y_train_resampled)
y_pred = rf_smote.predict(X_test)
